## C. Matriz de menciones entre medios
Construya una matriz de 5×5, donde cada fila y columna corresponden a un medio de prensa, y la entrada (i,j) contiene la cantidad de veces que el medio *i* menciona al medio *j*. \
\
**Opcional:** genere un grafo dirigido con esa matriz de adyacencia para visualizar las menciones. Puede ser útil la biblioteca `networkx`.

# Introducción a la Ciencia de Datos: Tarea 1

Este notebook contiene el código de base para realizar la Tarea 1 del curso. Puede copiarlo en su propio repositorio y trabajar sobre el mismo.
Las **instrucciones para ejecutar el notebook** están en la [página inicial del repositorio](https://gitlab.fing.edu.uy/maestria-cdaa/intro-cd).

Se utiliza el lenguaje Python y la librería Pandas. Si no tiene ninguna familiaridad con la librería, se recomienda realizar algún tutorial introductorio (ver debajo).
También se espera que los alumnos sean proactivos a la hora de consultar las documentaciones de las librerías y del lenguaje, para entender el código provisto.
Además de los recursos provistos en la [página del curso](https://eva.fing.edu.uy/course/view.php?id=1378&section=1), los siguientes recursos le pueden resultar interesantes:
 - [Pandas getting started](https://pandas.pydata.org/docs/getting_started/index.html#getting-started) y [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html): Son parte de la documentación en la página oficial de Pandas.
 - [Kaggle Learn](https://www.kaggle.com/learn): Incluye tutoriales de Python y Pandas.


Si desea utilizar el lenguaje R y está dispuesto a no utilizar (o traducir) este código de base, también puede hacerlo.

En cualquier caso, **se espera que no sea necesario revisar el código para corregir la tarea**, ya que todos los resultados y análisis relevantes deberían estar en el **informe en formato PDF**.

## Cargar bibliotecas (dependencias)
Recuerde instalar los requerimientos (`requirements.txt`) en el mismo entorno donde está ejecutando este notebook (ver [README](https://github.com/DonBraulio/introCD)).

In [ ]:
from time import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import nltk
import wordcloud
from datasets import load_dataset

## Descarga del dataset
En esta tarea se utilizará una base de datos abierta que contiene artículos de noticias publicados en distintos medios de prensa, con la finalidad de realizar una clasificación de textos según el medio de prensa al que pertenecen. [Link](https://huggingface.co/datasets/rjac/all-the-news-2-1-Component-one?utm_source=chatgpt.com) \
\
Ejecute la siguiente celda para descargar los datos y cargarlos en un dataframe de pandas. La constante `DATA_PATH` determina la ubicación donde se almacenaran los datos. \
\
El dataset entero pesa ~8.3gb. Para evitar demoras en la descarga/procesamiento vamos a utilizar el parámetro `streaming=True` y hacer un muestreo aleatorio para descargar una porción de los datos lo más representativa posible.

In [ ]:
ds = load_dataset("tomas-gr/all-the-news-2-1-Component-one-sampled", split="train",cache_dir="../data")
df = ds.to_pandas()

## Lectura de Datos

In [ ]:
# Veamos las primeras filas del DataFrame
df.head()

In [ ]:
# Veamos información general del DataFrame
df.info()

# Parte 1: Cargado y Limpieza de Datos

## A. Exploración de Datos
Analice el contenido del DataFrame. Reporte si existen datos faltantes en algún campo, o cualquier otro problema de calidad de datos que encuentre. \
En particular, analice la cantidad de artículos por medio de prensa, y a partir de este punto trabaje con los **cinco medios con mayor cantidad de artículos**.

In [ ]:
# TODO: Analice datos faltantes por columna
# Returns a count of missing values for every column
print("Cantidad de datos faltantes por columna:")
df.isna().sum()


In [ ]:
# TODO: Analice la cantidad de artículos por medio de prensa

# Tome los 5 medios con más artículos
top5 = df['publication'].dropna().value_counts().head(5).index
print('Los 5 medios con más artículos son:')
print(top5)
# top_5_publications = ...
df_top_5 = df[df['publication'].isin(top5)]
print('Cantidad de articulos de los 5 medios mas populares:')
len(df_top_5)

## B. Visualización temporal
Genere una gráfica que permita visualizar los artículos de los cinco medios a lo largo del tiempo, con alguna escala temporal adecuada. \
Comente si se identifican momentos de mayor actividad o patrones temporales en la cobertura.

In [ ]:
# TODO: Visualización de los artículos de cada medio a lo largo del tiempo
# Preste especial atención al formato de la columna 'date', ya que puede contener diferentes formatos de fecha.

df['date'] = pd.to_datetime(df['date'], errors='coerce', format= 'mixed')
df['date'].isna().sum()

# Recreate df_top_5 after date conversion to ensure dates are datetime objects
df_top_5 = df[df['publication'].isin(top5)]
df_top_5 = df_top_5[df_top_5['date'].notna()]
df_top_5['month'] = df_top_5['date'].dt.to_period('M')

counts = (
    df_top_5
    .groupby(['month', 'publication'])
    .size()
    .unstack(fill_value=0)
)

counts.index = counts.index.to_timestamp()

counts = counts[:-1]
counts.plot(figsize=(9,6), style='o-', markersize=4)
# Name axis, title and legend
plt.xlabel('Fecha')
plt.ylabel('Cantidad de artículos')
plt.title('Cantidad de artículos de los 5 medios más populares a lo largo del tiempo')
plt.legend(title='Medio de prensa')

Momento de mayor actividad: A partir de enero de 2020. Razones politicas.

## C. Limpieza de texto y conteo de palabras
Se provee la función `clean_text(...)` que realiza parte de la normalización del texto. \
**Complete la función** agregando signos de puntuación faltantes y cualquier otra normalización que considere oportuna. \
Compruebe el resultado observando el contenido del DataFrame procesado. Comente todas las transformaciones que haya agregado y justifique.

In [ ]:
def clean_text(df, column_name):

    # Eliminar primeras palabras hasta el primer "\n"
    result = df[column_name].str.replace(r"^[^\n]*\n", "", regex=True)

    # Convertir todo a minúsculas
    result = result.str.lower()

    # Handle common contractions before removing punctuation
    result = result.str.replace(r"I'm", "I am", regex=True)
    result = result.str.replace(r"you're", "you are", regex=True)
    result = result.str.replace(r"he's", "he is", regex=True)
    result = result.str.replace(r"she's", "she is", regex=True)
    result = result.str.replace(r"it's", "it is", regex=True)
    result = result.str.replace(r"we're", "we are", regex=True)
    result = result.str.replace(r"they're", "they are", regex=True)
    result = result.str.replace(r"I've", "I have", regex=True)
    result = result.str.replace(r"you've", "you have", regex=True)
    result = result.str.replace(r"we've", "we have", regex=True)
    result = result.str.replace(r"they've", "they have", regex=True)
    result = result.str.replace(r"can't", "cannot", regex=True)
    result = result.str.replace(r"won't", "will not", regex=True)
    result = result.str.replace(r"don't", "do not", regex=True)
    result = result.str.replace(r"doesn't", "does not", regex=True)

    # TODO: completar signos de puntuación faltantes (removed "'" since handled above)
    for punc in ["[", "\n", ",", ":", "?", "!", "(", ")", '"', "]"]:
        result = result.str.replace(punc, " ")

    return result

In [ ]:
# TODO: Aplique clean_text sobre la columna de texto elegida y cree una nueva columna "CleanText"

CleanText = clean_text(df_top_5, "article")

# Add CleanText as a new column to df_top_5
df_top_5['CleanText'] = CleanText
# print(df_top_5['CleanText'].head(n=5))
print(df_top_5.head(n=5))


Agregamos: "!", "(", ")", '"', "'", "]" pues pueden ser escritos enseguida luego de una palabra
Deducimos que "." no seria conveniente remover porque "U.S" podria ser confundido con el pronombre "us".

you minglei is a name but it can be confused with the pronoun "you"
is it ok to keep the reuters header?
some of them start with a space
maybe apply the clean_text to title?

## D. Elección de campos de texto
Discuta si conviene trabajar con:
- sólo el cuerpo del artículo,
- sólo el título,
- o una combinación de ambos.

Justifique brevemente su decisión.

*TODO: Escriba su análisis en el informe.*

Conviene trabjar con una combinacion de ambos pues la eleccion de palabras es muy importante en el titulo para captar la atencion del lector.

## E. Pistas que identifican al medio de prensa
Analice si en el texto aparecen pistas que identifiquen de manera directa al medio de prensa (nombres del medio, URLs, firmas, nombres de secciones, plantillas repetidas, etc.). \
En caso de encontrarlas, comente si considera conveniente eliminarlas o reducir su impacto, y justifique su decisión.

In [ ]:
# TODO: Explore el texto buscando pistas que identifiquen directamente al medio de prensa
# Por ejemplo, busque nombres de medios, URLs, firmas, etc.

La URL de todos las publicaciones indican el medio de prensa de forma directa.
El formato de la fecha da una indicacion pues cada medio utiliza un formato especifico.
Luego, existen otros indicadores en los titulos y articulos como se ve a continuacion:

In [ ]:
# Reuters

df_reuters = df[df['publication'] == 'Reuters']
print('Number of articles from Reuters: {}'.format(len(df_reuters)))

# En la seccion 'article'
df_reuters_with_word_reuters = df_reuters[df_reuters['article'].str.contains('Reuters', case=False, na=False)]
print('Number of publications from Reuters containing the word "Reuters" in the article: {}'.format(len(df_reuters_with_word_reuters)))

# En la seccion 'title'
df_reuters_with_word_brief = df_reuters[df_reuters['title'].str.contains('brief', case=False, na=False)]
print('Number of publications from Reuters containing the word "brief" in the title: {}'.format(len(df_reuters_with_word_brief)))

# Publicaciones con la palabra "Reuters" en el artículo y "brief" en el título

print('Articles from Reuters containing the word "Reuters" in the article')
display(df_reuters_with_word_reuters)

print('Articles from Reuters containing the word "brief" in the title')
display(df_reuters_with_word_brief)

In [ ]:
# People

df_people = df[df['publication'] == 'People']
display(df_people)

Parece ser que People solo escribe articulos sobre personas famosas. Los titulos siempre empiezan con el nombre y apellido de la persona en cuestion, o quizas un "How", "When" y luego el nombre de la persona.

Ademas, el titulo solo contiene palabras en mayuscula salvo conectores ("for", "and").

Se podria chequear ambas caracteristicas en los titulos y asi deducir que la publicacion viene de esta empresa.

O encontrar una lista de las personas mas mencionadas en los medios y usarla para descubir si alguna de las tres primeras palabras del articulo coinciden con un item de dicha lista.

In [ ]:
# CNBC
# Configure pandas to show full text content
pd.set_option('display.max_colwidth', 100)

df_cnbc = df[df['publication'] == 'CNBC']
# display(df_cnbc)

display(df_cnbc['article'])
display(len(df_cnbc))
df_cnbc_with_no_author = df_cnbc[df_cnbc['author'].isna()]
display(len(df_cnbc_with_no_author))

Observation: The word Reuters appear in some articles like referencing the other company:
(Updates with new first paragraph, adds background) March 29 (Reuters)

Casi ningun articulo tiene el autor especificado

## F. Restricción por sección o período temporal
Evalúe si conviene restringir el análisis a artículos de una misma sección temática o de un período temporal acotado, con el objetivo de reducir el efecto del tema sobre una futura tarea de clasificación por medio. \
No es necesario implementar esta restricción, pero sí discutir sus posibles ventajas y desventajas.

*TODO: Escriba su análisis en el informe.*

Si nuestro objetivo es crear un clasificador, conviene distinguir los articulos por tematica o periodo temporal. De lo contrario el clasificador podr'ia sesgarse e identificar siempre los articulos de una tematica o periodo a un determinado medio.
Lo ideal seria comparar articulos de una misma tematica o periodo temporal para entrenar a un clasificador.

# Parte 2: Conteo de Palabras y Visualizaciones

## A. Palabras más frecuentes por medio
Realice una visualización que permita comparar las palabras más frecuentes de cada uno de los cinco medios de prensa. \
Sin necesidad de implementarlo, proponga ideas para modificar esta visualización con el fin de encontrar diferencias entre secciones temáticas, fechas, o tipos de noticias.

GRAFICO DE COLUMNAS O MARIMEKKO PARA CADA MEDIO POR SEPARADO. PARA CADA MEDIO POR SEPARADO??? MAPA DE CALOR SERVIRIA???

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
import matplotlib.pyplot as plt
from wordcloud import WordCloud # Added to ensure WordCloud is imported

# Descargar recursos necesarios de nltk (solo la primera vez)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to fix the LookupError

# Función para obtener palabras más frecuentes
def get_most_frequent_words(text_series, n=20):
    # Combinar todo el texto
    all_text = ' '.join(text_series.dropna())

    # Tokenizar
    tokens = word_tokenize(all_text.lower())

    # Eliminar stopwords
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]

    # Contar frecuencias
    word_freq = Counter(filtered_tokens)

    return word_freq.most_common(n)

# Aplicar a cada medio
for publication in top5:
    df_pub = df_top_5[df_top_5['publication'] == publication]
    frequent_words = get_most_frequent_words(df_pub['CleanText'])

    print(f"\nPalabras más frecuentes en {publication}:")
    for word, freq in frequent_words:
        print(f"{word}: {freq}")

# Visualizacion de las 20 palabras mas frecuentes usando wordcloud, quitando las stopwords y signos de puntuacion

# Create custom stopwords that includes short words from contractions
custom_stopwords = set(stopwords.words('english'))
custom_stopwords.update(['m', 's', 're', 've', 'd', 'll', 't'])  # Short words from contractions

# Crear wordclouds separados para cada medio
for publication in top5:
    # Corrected line: Always filter from the original df_top_5
    df_pub = df_top_5[df_top_5['publication'] == publication]
    all_text = ' '.join(df_pub['CleanText'].dropna())

    # Crear wordcloud con mejores parámetros
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        max_words=20,  # Limitar a las 20 palabras más frecuentes
        colormap='viridis',
        prefer_horizontal=0.8,
        stopwords=custom_stopwords,  # Use custom stopwords with short words filtered
        min_word_length=2  # Exclude single letters like "u", "k", etc.
    ).generate(all_text)

    # Mostrar en figura separada
    plt.figure(figsize=(10, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(f'Word Cloud - {publication}', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()

Se podria considerar eliminar conectores como "also" y "like" y verbos ya que consideramos que los sustantivos y adjetivos demuestran mas lo que se quiere decir que este otro tipo de palabras.

## B. Medios con mayor cantidad de palabras
Corra el código que permite encontrar los medios con mayor cantidad de palabras. \
En caso de encontrar algún problema luego de realizar la visualización, comente a qué se debe y proponga formas de resolverlo.

In [ ]:
# Imprimir la cantidad de publicaciones totales por medio
for publication in top5:
    count = len(df_top_5[df_top_5['publication'] == publication])
    print(f"Total de publicaciones en {publication}: {count}")

# Imprimir la cantidad de palabras totales por medio
for publication in top5:
    df_pub = df_top_5[df_top_5['publication'] == publication]
    total_words = df_pub['CleanText'].str.split().str.len().sum()
    print(f"Total de palabras en {publication}: {total_words}")

# Imprimir la cantidad promedio de palabras por artículo por medio
for publication in top5:
    df_pub = df_top_5[df_top_5['publication'] == publication]
    avg_words = df_pub['CleanText'].str.split().str.len().mean()
    print(f"Promedio de palabras por artículo en {publication}: {avg_words:.2f}")

## C. Matriz de menciones entre medios
Construya una matriz de 5×5, donde cada fila y columna corresponden a un medio de prensa, y la entrada (i,j) contiene la cantidad de veces que el medio *i* menciona al medio *j*. \
\
**Opcional:** genere un grafo dirigido con esa matriz de adyacencia para visualizar las menciones. Puede ser útil la biblioteca `networkx`.

In [ ]:
import pandas as pd
import numpy as np
import re

# Create a mapping from original publication names to search terms
# 'People' is capitalized for case-sensitive matching as requested.
search_terms = {
    'Reuters': 'Reuters',
    'The New York Times': 'The New York Times',
    'CNBC': 'CNBC',
    'The Hill': 'The Hill',
    'People': 'People' # Search for 'People' with capital 'P'
}

# Initialize a 5x5 matrix with zeros, with publication names as index and columns
mentions_matrix = pd.DataFrame(0, index=top5, columns=top5)

# Iterate through each source publication (row in the matrix)
for source_pub in top5:
    # Filter articles for the current source publication, using the original 'article' text
    # to preserve case for 'People' matching.
    source_articles_text = df_top_5[df_top_5['publication'] == source_pub]['article'].dropna()

    # Iterate through each target publication (column in the matrix)
    for target_pub in top5:
        target_search_term = search_terms[target_pub]

        # Determine the regex pattern based on whether it's a single word or multi-word phrase
        if ' ' in target_search_term: # Multi-word phrase, use literal search
            pattern = re.escape(target_search_term)
        else: # Single word, use word boundaries to avoid partial matches (e.g., 'hill' in 'hilly')
            pattern = r'\b' + re.escape(target_search_term) + r'\b'

        # Conditionally apply IGNORECASE: only 'People' should be case-sensitive
        current_flags = 0
        if target_pub != 'People':
            current_flags = re.IGNORECASE

        # Count occurrences of target_pub's search term in source_pub's articles
        mention_count = source_articles_text.str.count(pattern, flags=current_flags).sum()

        mentions_matrix.loc[source_pub, target_pub] = mention_count

print("Matriz de menciones entre medios:")
display(mentions_matrix)

print("\nNota: El conteo de menciones para 'People' ahora es case-sensitive, only counting 'People' (magazine name).")
print("Other publication mentions are still case-insensitive.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a heatmap of the mentions matrix using seaborn
plt.figure(figsize=(10, 8)) # Set a good figure size
sns.heatmap(
    mentions_matrix,
    annot=True,     # Annotate cells with the data value
    fmt='d',        # Format annotations as integers
    cmap='Blues', # Choose a blue color map
    linewidths=.5,  # Add lines between cells
    cbar_kws={'label': 'Cantidad de Menciones'}
)

plt.title('Matriz de Menciones entre Medios de Prensa', size=16) # Add a title
plt.xlabel('Medio Mencionado', size=12) # X-axis label
plt.ylabel('Medio Fuente', size=12)    # Y-axis label
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
plt.yticks(rotation=0)             # Ensure y-axis labels are horizontal
plt.tight_layout() # Adjust layout to prevent labels from being cut off

# Save the figure as a PDF before calling plt.show()
plt.savefig('mentions_matrix_heatmap.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Create a directed graph from the adjacency matrix
G = nx.DiGraph(mentions_matrix.values)

# Map node indices to publication names for better visualization
mapping = {i: pub for i, pub in enumerate(mentions_matrix.index)}
G = nx.relabel_nodes(G, mapping)

# Draw the graph
plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G, k=1.0, iterations=50) # Increased k for more spacing between nodes

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=4000, alpha=0.9)

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

# Draw edges, scaling width by mention count
edges = [(u, v, d['weight']) for u, v, d in G.edges(data=True)]
edge_weights = [d['weight'] for u, v, d in G.edges(data=True)]

# Scale edge widths for better visualization (min_width=0.5, max_width=5.0)
max_weight = max(edge_weights) if edge_weights else 1
min_edge_width = 0.5
max_edge_width = 5.0
scaled_widths = [min_edge_width + (w / max_weight) * (max_edge_width - min_edge_width) for w in edge_weights]

# Draw curved edges for better visibility of multiple edges between nodes
nx.draw_networkx_edges(G, pos, width=scaled_widths, arrowsize=20, alpha=1.0, edge_color='black', connectionstyle='arc3,rad=0.2') # Increased rad for more pronounced curves

# Add edge labels (mention counts) only if there are edges
if edges:
    edge_labels = {(u, v): d['weight'] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=10)

plt.title("Grafo Dirigido de Menciones entre Medios de Prensa", size=16)
plt.axis('off') # Hide axes
plt.tight_layout() # Adjust layout
plt.savefig('grafo dirigido.pdf', bbox_inches='tight')
plt.show()

## D. Preguntas propuestas
Proponga al menos tres preguntas que se podrían intentar responder a partir de estos datos, y mencione posibles caminos para responderlas, sin implementar nada.

*TODO: Escriba sus preguntas y posibles caminos en el informe.*

Que medio es mas mencionado por otros?
Que medio menciona mas otros medios?
Que medio se autorreferncia mas? Es autoreferencia o simplemente tienen un template con su nombre?
